# Fine-tuning — Classification par domaine juridique (FR + AR), version ColabAssistant juridique ADLI Morocco — classifieur par domaine sur le corpus BulletinOfficiel du Royaume du Maroc, en français **et** arabe.Ce notebook :1. calcule la **baseline mots-clés** (référence : `keyword_classifier.py` du dépôt) ;2. fine-tune **`xlm-roberta-base`** avec validation croisée stratifiée (5 folds,   perte pondérée par classe) ;3. produit une **évaluation quantitative** : accuracy, F1 macro par domaine, par   langue, matrice de confusion, comparaison transformer vs mots-clés ;4. entraîne le **modèle final** (100 % des données) pour le déploiement ;5. télécharge (`files.download`) le ZIP du modèle + le rapport d'évaluation JSON.**Pense à activer un GPU** : `Exécution > Modifier le type d'exécution > GPU (T4)`.Fichier de données requis : `domain_dataset_final.csv` (à uploader, ou monter Drive).

In [ ]:
# Cellule d'avertissement : après exécution de l'installation, REDÉMARRE la# session (Exécution > Redémarrer la session), PUIS relance le notebook depuis# ici. transformers doit être réimporté à froid (sinon erreur# 'Could not import module Trainer' due au conflit torch/torchvision).!pip install -q --upgrade transformers datasets accelerate scikit-learn pandas!pip uninstall -y -q torchvision > /dev/null 2>&1print("Installation OK — redémarre la session maintenant (Exécution > Redémarrer la session).")

## 1. Chargement du jeu de donnéesChoisis **une** des deux options (upload manuel ou Google Drive).

In [ ]:
# --- Option A : upload manuel ---from google.colab import filesimport jsonuploaded = files.upload()csv_path = list(uploaded.keys())[0]print("Fichier chargé :", csv_path)

In [ ]:
# --- Option B : Google Drive (décommente) ---# from google.colab import drive# drive.mount('/content/drive')# csv_path = '/content/drive/MyDrive/domain_dataset_final.csv'

In [ ]:
import pandas as pddf = pd.read_csv(csv_path, encoding='utf-8-sig')print(f"{len(df)} lignes chargées — colonnes : {list(df.columns)}")df['label'].value_counts()

## 2. Baseline mots-clés (référence)Copie autonome du classifieur par mots-clés du dépôt (`src/classification/keyword_classifier.py`),pour une comparaison équitable sur le même protocole de validation croisée.

In [ ]:
import reDOMAIN_KEYWORDS = { "Fiscal": {"fr": ["impôt","taxe","TVA","IR","IS","douane","recouvrement","exonération","déduction","fiscal","contribution","patente"],            "ar": ["ضريبة","رسم","جباية","الضرائب","الاستخلاص","الإعفاء","الخصم","الضريبة على القيمة المضافة"]}, "Social": {"fr": ["travail","salarié","congé","retraite","assurance","maladie","accident","chômage","sécurité sociale","allocations","pension"],            "ar": ["الشغل","أجير","عطلة","تقاعد","تأمين","مرض","حادثة","بطالة","الضمان الاجتماعي","منح","معاش"]}, "Administratif": {"fr": ["fonctionnaire","administration","collectivité","territoriale","élection","municipal","préfecture","province","commune","établissement public","agent","décentralisation"],            "ar": ["موظف","إدارة","جماعة","ترابية","انتخاب","جماعي","عمالة","إقليم","المغرب","مؤسسة عمومية","لا مركزية"]}, "Civil": {"fr": ["contrat","mariage","succession","testament","donation","divorce","héritage","tutelle","curatelle","propriété"],            "ar": ["عقد","زواج","إرث","وصية","هبة","طلاق","ميراث","وصاية","المدني","ملكية"]}, "Pénal": {"fr": ["infraction","délit","crime","peine","emprisonnement","amende","tribunal","cour d'assises","procès","enquête"],            "ar": ["جريمة","جنحة","مخالفة","عقوبة","سجن","غرامة","محكمة","جنايات","محاكمة","تحقيق"]}, "Commercial": {"fr": ["commerce","société","entreprise","commerçant","fonds de commerce","registre du commerce","achat","vente","contractant","fournisseur"],            "ar": ["تجارة","شركة","مقاولة","تاجر","محل تجاري","السجل التجاري","شراء","بيع","متعاقد","مورد"]}, "Environnement": {"fr": ["environnement","eau","air","déchet","pollution","protection","foret","biodiversité","climat","énergie","développement durable"],            "ar": ["البيئة","الماء","الهواء","نفاية","تلوث","حماية","غابة","التنوع البيولوجي","المناخ","الطاقة","التنمية المستدامة"]}, "Urbain": {"fr": ["urbanisme","logement","construction","aménagement","ville","métropole","architecte","permis de construire","zoning"],            "ar": ["التعمير","سكن","بناء","تهيئة","مدينة","متروبول","مهندس معماري","رخصة البناء","التقسيم"]},}def classify_text_with_scores(text: str, lang: str = "fr"):    text_lower = text.lower(); scores = {}    for domain, lang_kw in DOMAIN_KEYWORDS.items():        score = 0        for word in lang_kw[lang]:            pattern = r"\b" + re.escape(word.lower()) + r"\b"            if word == "بناء":                pattern += r"(?!\s+على)"            score += len(re.findall(pattern, text_lower))        scores[domain] = score    return scoresdef classify_text(text: str, lang: str = "fr"):    scores = classify_text_with_scores(text, lang)    best_domain, best_score = max(scores.items(), key=lambda item: item[1])    return "Indéterminé" if best_score == 0 else best_domain

## 3. Filtrage et préparation (identique aux deux modèles)- Retrait d'`Indéterminé` ; classes avec < `MIN_SAMPLES_PER_CLASS` exemplaires exclues ;- encodage des labels ; tokenizer `xlm-roberta-base`, max 256 tokens.

In [ ]:
MIN_SAMPLES_PER_CLASS = 10data = df[df['label'] != 'Indéterminé'].copy()counts = data['label'].value_counts()kept = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()dropped = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()print("Domaines exclus (trop peu d'exemples) :", sorted(dropped) if dropped else "aucun")data = data[data['label'].isin(kept)].reset_index(drop=True)print(f"{len(data)} exemples utilisables, {len(kept)} domaines : {kept}")print(f"  FR : {(data['lang']=='fr').sum()}  |  AR : {(data['lang']=='ar').sum()}")print(data['label'].value_counts())

In [ ]:
from sklearn.preprocessing import LabelEncoderlabel_encoder = LabelEncoder()data['label_id'] = label_encoder.fit_transform(data['label'])id2label = {i: l for i, l in enumerate(label_encoder.classes_)}label2id = {l: i for i, l in id2label.items()}num_labels = len(id2label)print(id2label)

In [ ]:
import torchfrom transformers import AutoTokenizerMODEL_NAME = "xlm-roberta-base"MAX_LENGTH = 256tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)def tokenize(texts):    return tokenizer(list(texts), padding="max_length", truncation=True,                     max_length=MAX_LENGTH, return_tensors="pt")

## 4. Baseline mots-clés — protocole 5-fold identique au transformerOn réutilise le **même découpage** (`StratifiedKFold`, seed 42) pour comparermots-clés vs transformer à périmètre strictement égal.

In [ ]:
import numpy as npfrom sklearn.model_selection import StratifiedKFoldfrom sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrixskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)splits = list(skf.split(data['text'].to_numpy(), data['label_id'].to_numpy()))# --- Baseline mots-clés sur les mêmes folds ---kw_y_true, kw_y_pred = [], []for _, test_idx in splits:    for i in test_idx:        pred = classify_text(data.loc[i, 'text'], lang=data.loc[i, 'lang'])        kw_y_true.append(data.loc[i, 'label_id'])        kw_y_pred.append(label2id.get(pred, -1))  # domaine non connu -> -1kw_acc = accuracy_score(kw_y_true, kw_y_pred)kw_f1 = f1_score(kw_y_true, kw_y_pred, average="macro",                 labels=list(range(num_labels)), zero_division=0)print("Baseline mots-clés (5-fold cumulé) :")print(f"  accuracy : {kw_acc:.3f}")print(f"  F1 macro : {kw_f1:.3f}")print()print("Rapport par domaine (mots-clés) :")print(classification_report(kw_y_true, kw_y_pred, labels=list(range(num_labels)),      target_names=[id2label[i] for i in range(num_labels)], zero_division=0, digits=3))

## 5. Dataset PyTorch + Trainer pondéré

In [ ]:
from torch.utils.data import Datasetimport torch.nn as nnclass DomainDataset(Dataset):    def __init__(self, texts, labels):        self.encodings = tokenize(texts)        self.labels = torch.tensor(labels, dtype=torch.long)    def __len__(self): return len(self.labels)    def __getitem__(self, idx):        item = {k: v[idx] for k, v in self.encodings.items()}        item["labels"] = self.labels[idx]        return itemfrom transformers import Trainerclass WeightedTrainer(Trainer):    def __init__(self, class_weights=None, **kwargs):        super().__init__(**kwargs)        self.class_weights = class_weights    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):        labels = inputs.pop("labels")        outputs = model(**inputs)        logits = outputs.logits        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None        loss_fct = nn.CrossEntropyLoss(weight=weight)        loss = loss_fct(logits, labels)        return (loss, outputs) if return_outputs else lossdef compute_metrics(eval_pred):    logits, labels = eval_pred    preds = np.argmax(logits, axis=1)    return {"accuracy": accuracy_score(labels, preds),            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0)}

## 6. Validation croisée stratifiée (5 folds) du transformerEntraîne 5 modèles sur GPU T4 (~5-10 min). On ne garde que l'évaluation par fold ;le **modèle de déploiement** est entraîné plus bas sur 100 % des données.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArgumentsimport gcEPOCHS = 8BATCH_SIZE = 16LEARNING_RATE = 2e-5fold_accs, fold_f1s = [], []tr_y_true, tr_y_pred = [], []for fold, (train_idx, test_idx) in enumerate(splits):    model = AutoModelForSequenceClassification.from_pretrained(        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)    train_ds = DomainDataset(data.loc[train_idx, 'text'], data.loc[train_idx, 'label_id'])    test_ds = DomainDataset(data.loc[test_idx, 'text'], data.loc[test_idx, 'label_id'])    counts = np.bincount(data.loc[train_idx, 'label_id'], minlength=num_labels)    cw = torch.tensor([len(train_idx) / (num_labels * max(c, 1)) for c in counts], dtype=torch.float)    args = TrainingArguments(        output_dir=f"/content/fold_{fold}",        num_train_epochs=EPOCHS,        per_device_train_batch_size=BATCH_SIZE,        learning_rate=LEARNING_RATE,        weight_decay=0.01,        save_strategy="no",        logging_strategy="no",        eval_strategy="epoch",        report_to="none",        seed=42)    trainer = WeightedTrainer(class_weights=cw, model=model, args=args,                              train_dataset=train_ds, eval_dataset=test_ds,                              compute_metrics=compute_metrics)    trainer.train()    m = trainer.evaluate()    fold_accs.append(m["eval_accuracy"])    fold_f1s.append(m["eval_f1_macro"])    print(f"  fold {fold}: acc {m['eval_accuracy']:.3f} / f1_macro {m['eval_f1_macro']:.3f}")    preds = trainer.predict(test_ds).predictions    pred_ids = np.argmax(preds, axis=1)    tr_y_true.extend(data.loc[test_idx, 'label_id'].tolist())    tr_y_pred.extend(pred_ids.tolist())    del model, trainer; gc.collect()    if torch.cuda.is_available(): torch.cuda.empty_cache()print()print(f"Accuracy moyenne  : {np.mean(fold_accs):.3f} (+/- {np.std(fold_accs):.3f})")print(f"F1 macro moyen    : {np.mean(fold_f1s):.3f} (+/- {np.std(fold_f1s):.3f})")print()print("Rapport par domaine (transformer, folds cumulées) :")print(classification_report(tr_y_true, tr_y_pred, labels=list(range(num_labels)),      target_names=[id2label[i] for i in range(num_labels)], zero_division=0, digits=3))tr_f1 = f1_score(tr_y_true, tr_y_pred, average="macro",                 labels=list(range(num_labels)), zero_division=0)print(f"\nComparaison F1 macro : mots-clés {kw_f1:.3f}  vs  transformer {tr_f1:.3f}")

## 7. Matrice de confusion — transformer (folds cumulées)

In [ ]:
import matplotlib.pyplot as pltimport itertoolscm = confusion_matrix(tr_y_true, tr_y_pred, labels=list(range(num_labels)))plt.figure(figsize=(9, 7))plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)plt.title("Transformer — matrice de confusion (5-fold cumulées)")plt.colorbar()tick_marks = list(range(num_labels))plt.xticks(tick_marks, [id2label[i] for i in tick_marks], rotation=45, ha="right")plt.yticks(tick_marks, [id2label[i] for i in tick_marks])thresh = cm.max() / 2.for i, j in itertools.product(range(num_labels), range(num_labels)):    plt.text(j, i, format(cm[i, j], "d"),             horizontalalignment="center",             color="white" if cm[i, j] > thresh else "black")plt.ylabel("Vrai domaine"); plt.xlabel("Prédit")plt.tight_layout(); plt.show()

## 8. Modèle final (déploiement) + inférenceEntraîné sur **100 % des données** (train + val réunis) — ne pas juger sa qualitésur ses propres données : se référer aux scores de validation croisée ci-dessus.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArgumentsfull_ds = DomainDataset(data['text'], data['label_id'])counts = np.bincount(data['label_id'], minlength=num_labels)cw = torch.tensor([len(data) / (num_labels * max(c, 1)) for c in counts], dtype=torch.float)final_model = AutoModelForSequenceClassification.from_pretrained(    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)final_args = TrainingArguments(    output_dir="/content/final_model",    num_train_epochs=EPOCHS,    per_device_train_batch_size=BATCH_SIZE,    learning_rate=LEARNING_RATE,    weight_decay=0.01,    save_strategy="no",    logging_strategy="epoch",    report_to="none",    seed=42)final_trainer = WeightedTrainer(class_weights=cw, model=final_model, args=final_args,                                train_dataset=full_ds, compute_metrics=compute_metrics)final_trainer.train()

In [ ]:
import json as _jsonSAVE_DIR = "/content/domain_classifier_fr_ar"final_trainer.save_model(SAVE_DIR)tokenizer.save_pretrained(SAVE_DIR)with open(f"{SAVE_DIR}/id2label.json", "w", encoding="utf-8") as f:    _json.dump(id2label, f, ensure_ascii=False, indent=2)print("Modèle sauvegardé dans", SAVE_DIR)def classify_text_transformer(text: str) -> dict:    device = next(final_model.parameters()).device    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)    inputs = {k: v.to(device) for k, v in inputs.items()}    with torch.no_grad():        logits = final_model(**inputs).logits    probs = torch.softmax(logits, dim=1)[0]    pred_id = int(torch.argmax(probs))    return {"label": id2label[pred_id],            "scores": {id2label[i]: float(probs[i]) for i in range(num_labels)}}

In [ ]:
samples = [ "Est fixé dans l'annexe au présent arrêté le cahier des charges relatif aux " "spécifications techniques minimales des infrastructures de télécommunications.", "Le montant de l'amende de transaction est fixé par l'administration des eaux " "et forêts en fonction de la gravité de l'infraction constatée.", "يخضع المفوض القضائي كل سنة لدورة على الاقل من دورات التكوين المستمر.", "تحدث لجنة علمية بالمعهد تتولى ابداء الراي في البرامج البيداغوجية وموضوعات " "البحث العلمي.",]for s in samples:    r = classify_text_transformer(s)    top3 = sorted(r["scores"].items(), key=lambda kv: kv[1], reverse=True)[:3]    print(f"Texte : {s[:70]}...")    print(f"  -> Transformer : {r['label']}  (top 3 : {top3})")    print()

## 9. Rapport d'évaluation (JSON) + téléchargementsProduit un rapport consolidé, puis télécharge :- `domain_classifier_fr_ar.zip` — le modèle final (à redéposer dans `models/`) ;- `eval_domaine_classifieur.json` — accuracy, F1 par domaine, comparaison  mots-clés vs transformer, matrice de confusion.

In [ ]:
report = {  "model": MODEL_NAME,  "n_examples": int(len(data)),  "domains": {id2label[i]: int((data['label_id']==i).sum()) for i in range(num_labels)},  "baseline_keywords": {"accuracy": round(kw_acc, 4), "f1_macro": round(kw_f1, 4)},  "transformer_cv": {      "accuracy_mean": round(float(np.mean(fold_accs)), 4),      "accuracy_std": round(float(np.std(fold_accs)), 4),      "f1_macro_mean": round(float(np.mean(fold_f1s)), 4),      "f1_macro_std": round(float(np.std(fold_f1s)), 4),  },  "transformer_per_domain": classification_report(      tr_y_true, tr_y_pred, labels=list(range(num_labels)),      target_names=[id2label[i] for i in range(num_labels)],      zero_division=0, output_dict=True),  "confusion_train": cm.tolist(),}with open("/content/eval_domaine_classifieur.json", "w", encoding="utf-8") as f:    _json.dump(report, f, ensure_ascii=False, indent=2)print("Rapport JSON écrit.")!zip -rq /content/domain_classifier_fr_ar.zip /content/domain_classifier_fr_arfrom google.colab import filesfiles.download("/content/domain_classifier_fr_ar.zip")files.download("/content/eval_domaine_classifieur.json")